Olá, **Luiz**!

Meu nome é **Gabriel Aguiar** e serei o revisor do seu projeto hoje.

Quando identificar um erro pela primeira vez, apenas vou apontá-lo para que você mesmo possa encontrá-lo e corrigi-lo. Ao longo do notebook também farei observações de melhoria e comentários sobre suas análises — meu objetivo é prepará-lo(a) para trabalhar como **Data Analyst** no mercado real.

Se não conseguir identificar o erro, na próxima iteração darei uma dica mais precisa com exemplos práticos. Estou aberto a feedbacks e discussões.

**Por favor, não mova, modifique ou exclua meus comentários.**

Você encontrará meus comentários em caixas coloridas:

<div class="alert alert-block alert-success">
<b>Comentário do revisor (Gabriel Aguiar):</b> <a class="tocSkip"></a>
Sucesso. Tudo foi feito corretamente.
</div>

<div class="alert alert-block alert-warning">
<b>Comentário do revisor (Gabriel Aguiar):</b> <a class="tocSkip"></a>
Atenção. Sugestão ou observação de melhoria.
</div>

<div class="alert alert-block alert-danger">
<b>Comentário do revisor (Gabriel Aguiar):</b> <a class="tocSkip"></a>
Precisa de correção. O projeto não pode ser aceito com comentários em vermelho abertos.
</div>

Você pode me responder usando:

<div class="alert alert-block alert-info">
<b>Resposta do Aluno.</b> <a class="tocSkip"></a>
</div>

# Decomposição de Tarefas — Caso Principal: CallMeMaybe

- **Analista:** Luiz Almeida
- **Sprint:** 14 — Projeto Final (TripleTen)
- **Data:** 17/09/2026


Este documento apresenta a decomposição da tarefa analítica do Caso Principal. O objetivo é planejar, de forma estruturada e verificável, todas as etapas necessárias para identificar **operadores ineficientes** no serviço de telefonia virtual **CallMeMaybe**, bem como validar estatisticamente as hipóteses que sustentam essa identificação. As etapas aqui descritas serão implementadas na fase seguinte do projeto.

<div class="alert alert-block alert-success">
<b>Comentário geral do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Luiz, parabéns! Sua decomposição está aprovada. Você entregou uma contextualização de negócio sólida, uma descrição de dados completa e cinco hipóteses bem formuladas, com testes estatísticos escolhidos com justificativa. O plano de ação está detalhado o suficiente pra ser executado sem retrabalho.

Deixei um comentário opcional sobre correção de múltiplos testes (não é ensinado no curso, fica como curiosidade). Fique à vontade para avançar para a implementação. Qualquer dúvida, conte comigo. Até breve!
</div>

## 1. Identificação do Problema de Negócio

### 1.1 Contexto
- A **CallMeMaybe** opera como uma plataforma de telefonia virtual (VoIP) voltada para o segmento B2B. Seus clientes são organizações que gerenciam grandes volumes de chamadas recebidas e realizadas por meio de múltiplos operadores. A rede também permite chamadas internas entre operadores de um mesmo cliente.

### 1.2 O problema
- A empresa está desenvolvendo uma nova funcionalidade que permitirá aos **supervisores** identificar, dentro de suas equipes, os operadores com desempenho abaixo do esperado. Segundo o enunciado, um operador é considerado **ineficiente** quando apresenta pelo menos uma das seguintes características:

1. **Alto volume de chamadas recebidas perdidas** — sejam internas ou externas;
2. **Tempo de espera prolongado** nas chamadas recebidas;
3. **Baixo volume de chamadas ativas de saída**, no caso de operadores cuja função principal é originar chamadas.

### 1.3 Por que isso importa
Sem uma forma objetiva de medir ineficiência, os supervisores tomam decisões (treinamento, realocação, ajuste de metas, desligamento) com base em percepção subjetiva. Isso gera:
- **Custo operacional elevado** — operadores ineficientes consomem recursos sem retorno proporcional;
- **Insatisfação dos clientes finais** — chamadas perdidas e tempos de espera longos degradam a experiência das organizações contratantes;
- **Dificuldade de priorização** — sem KPIs claros, é impossível comparar operadores entre si ou ao longo do tempo.

### 1.4 Objetivo analítico
- Realizar uma **análise exploratória de dados (EDA)** nos dois datasets fornecidos, **identificar padrões mensuráveis que caracterizem operadores ineficientes** e **testar hipóteses estatísticas** que sustentem a classificação. O resultado será comunicado em dois formatos: um **dashboard interativo** e um **relatório final no formato CAR** (Challenge, Actions, Results), ambos voltados a supervisores e gestores de operações.

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Boa, você conectou o problema técnico (identificar operadores ineficientes) ao impacto real de negócio (custo operacional, satisfação do cliente, dificuldade de priorização). Essa contextualização é o que diferencia um projeto que só calcula números de um que sustenta uma decisão de negócio.
</div>

## 2. Descrição dos Dados
- A análise utilizará **dois datasets complementares**, fornecidos pela CallMeMaybe e relacionados pela coluna `user_id`.

### 2.1 `telecom_dataset_new.csv` — dados de uso (~3,3 MB)
Contém estatísticas **diárias de uso**, agrupadas por cliente, data, direção da chamada e operador.

| Coluna | Tipo esperado | Descrição |
|---|---|---|
| `user_id` | int | ID da conta do cliente (organização) |
| `date` | datetime | Data em que as estatísticas foram coletadas |
| `direction` | string | Direção da chamada: `in` (recebida) ou `out` (realizada) |
| `internal` | bool | Indica se a chamada foi interna (entre operadores do mesmo cliente) |
| `operator_id` | int / float | ID do operador; esperado **vazio** em chamadas recebidas externas |
| `is_missed_call` | bool | Indica se foi uma chamada perdida |
| `calls_count` | int | Número de chamadas no registro |
| `call_duration` | int | Duração da chamada (sem tempo de espera) |
| `total_call_duration` | int | Duração total (incluindo tempo de espera) |

### 2.2 `telecom_clients.csv` — dados cadastrais (~14 KB)

| Coluna | Tipo esperado | Descrição |
|---|---|---|
| `user_id` | int | ID do cliente |
| `tariff_plan` | string | Plano tarifário atual (`A`, `B` ou `C`) |
| `date_start` | date | Data de registro do cliente |

### 2.3 Volume e qualidade esperada
- **Volume:** será confirmado com `.shape` para ambos os datasets.
- **Qualidade esperada:**
  - Presença de `operator_id` ausente em chamadas recebidas externas (`direction='in'` e `internal=False`);
  - Possíveis **registros duplicados** no dataset de uso, a serem investigados e tratados;
  - Necessidade de conversão de `date` para `datetime` e de `is_missed_call` / `internal` para `bool`;
  - Verificação de nulos em todas as colunas, com decisão documentada para cada caso.

### 2.4 Estatísticas descritivas planejadas
- `.describe()` para `calls_count`, `call_duration` e `total_call_duration`;
- Tabelas de frequência para `direction`, `internal`, `is_missed_call` e `tariff_plan`;
- Distribuições (histogramas e boxplots) para variáveis contínuas.

### 2.5 Variáveis-chave

| Variável | Papel na análise |
|---|---|
| `operator_id` | Unidade de análise principal (operador) |
| `is_missed_call` | Métrica central de ineficiência |
| `direction` | Separação entre operadores de entrada e saída |
| `internal` | Contexto da chamada |
| `calls_count` | Volume de atividade do operador |
| `call_duration` | Duração efetiva (produtividade) |
| `total_call_duration` | Duração total (inclui espera → base do cálculo de `wait_time`) |
| `tariff_plan` | Segmentação dos clientes |
| `date_start` | Antiguidade do cliente (variável derivada: tempo de casa) |

### 2.6 Estratégia de integração
Os dois datasets serão unidos por **`user_id`** (left join do dataset de uso com o cadastral). Isso permitirá enriquecer a análise de comportamento dos operadores com **plano tarifário** e **antiguidade do cliente**, viabilizando segmentações que respondem diretamente às hipóteses H4 e H5.

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Boa, a descrição dos dados está bem completa: colunas, tipos esperados, qualidade esperada (nulos, duplicatas) e até a estratégia de merge entre os dois datasets. Isso já dá pra qualquer pessoa da equipe entender o que vai ser trabalhado sem precisar abrir os CSVs.
</div>

## 3. Hipóteses a Serem Validadas
As hipóteses abaixo são **mensuráveis com os dados disponíveis** e serão testadas por métodos estatísticos rigorosos. Cada uma está formulada no formato **nula (H0) / alternativa (H1)** sempre que aplicável.

### H1 — Tempo de espera e chamadas perdidas
- **H0:** Não há associação entre tempo médio de espera e taxa de chamadas perdidas por operador.
- **H1:** Operadores com maior tempo médio de espera apresentam maior taxa de chamadas perdidas.

### H2 — Operadores de saída com baixo volume
- **H0:** O volume médio de chamadas ativas de saída dos operadores classificados como ineficientes é igual à média geral.
- **H1:** Operadores de saída com baixo volume médio de chamadas ativas diferem significativamente da média geral.

### H3 — Chamadas internas vs. externas
- **H0:** O tempo médio de espera é igual entre chamadas internas e externas.
- **H1:** Chamadas internas apresentam tempo médio de espera **menor** que chamadas externas.

### H4 — Diferenças por plano tarifário
- **H0:** A taxa de chamadas perdidas é igual entre clientes dos planos A, B e C.
- **H1:** Pelo menos um dos planos apresenta taxa de chamadas perdidas significativamente diferente dos demais.

### H5 — Antiguidade do cliente
- **H0:** Não há associação entre tempo de casa do cliente e taxa de chamadas perdidas de seus operadores.
**H1:** Clientes mais antigos apresentam menor taxa de chamadas perdidas de seus operadores.

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Boa, todas as cinco hipóteses estão formuladas corretamente no formato H0/H1, cada uma mensurável com as colunas que você descreveu na seção 2. Isso é o que torna a decomposição verificável de verdade, e não só uma lista de intenções.
</div>

## 4. Indicadores-Chave (KPIs)
Todos os KPIs serão calculados **por operador**, **por direção** e **por plano tarifário**, conforme a hipótese em teste.

| KPI | Fórmula | Unidade | Utilização |
|---|---|---|---|
| **Taxa de chamadas perdidas** | `Σ calls_count (is_missed_call=True) / Σ calls_count (total)` | % | H1, H4, H5 |
| **Tempo médio de espera (`wait_time`)** | `média(total_call_duration − call_duration)` | segundos | H1, H3 |
| **Volume médio de chamadas de saída** | `média(calls_count)` com `direction='out'` | chamadas | H2 |
| **Duração média efetiva** | `média(call_duration)` com `is_missed_call=False` | segundos | Caracterização de operadores |
| **Proporção de chamadas internas** | `Σ calls_count (internal=True) / Σ calls_count (total)` | % | Contexto |
| **Nº de operadores ativos por cliente** | `nunique(operator_id)` agrupado por `user_id` | contagem | Contexto |
| **Tempo de casa do cliente** | `data_atual − date_start` | dias | H5 |

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Boa, definir a fórmula de cada KPI e amarrar cada um à hipótese que ele vai sustentar é o tipo de rigor que evita retrabalho depois: quando você for calcular de fato, já sabe exatamente o que precisa produzir.
</div>

## 5. Ações a Executar
O plano de ação está dividido em cinco blocos: preparação e tratamento dos dados; análise dos dados; validação das hipóteses; comunicação dos resultados; e estimativa de tempo. Cada bloco contém atividades numeradas, executáveis e verificáveis.

### 5.1 Preparação e Tratamento dos Dados

**A1.** Carregar `telecom_dataset_new.csv` e `telecom_clients.csv` com Pandas e inspecionar formato, tipos e volume (`.shape`, `.info()`, `.head()`, `.sample()`).

**A2.** Verificar e tratar **valores ausentes** em ambos os datasets, com atenção especial a `operator_id` (esperado vazio em chamadas `in` externas). Documentar a decisão para cada coluna com nulos.

**A3.** Verificar e tratar **registros duplicados** no dataset de uso — investigar se são duplicatas reais ou repetições legítimas (ex.: mesmo dia, mesmo operador, mesma direção), e decidir pela remoção ou agregação.

**A4.** **Converter tipos**: `date` para `datetime`, `is_missed_call` e `internal` para `bool`, `operator_id` para `Int64` (nullable) quando aplicável.

**A5.** Tratar `operator_id` ausente nas chamadas recebidas externas — criar categoria `'external'` ou manter como nulo, com justificativa documentada.

**A6.** Padronizar **nomes de colunas** conforme a descrição oficial, caso haja divergência.

**A7.** Realizar o **merge** dos dois datasets por `user_id` (left join do uso com o cadastro), preservando todos os registros do dataset principal.

**A8.** Criar **colunas derivadas**:
- `wait_time = total_call_duration − call_duration`
- `tenure_days = data_referência − date_start`
- `month = date.dt.to_period('M')`

**A9.** Salvar uma cópia intermediária do dataset tratado (`telecom_merged.parquet`) para garantir reprodutibilidade da análise.

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Boa, o plano de tratamento dos dados está bem granular e antecipa problemas reais desse dataset (nulos esperados em `operator_id`, duplicatas, conversão de tipos). Salvar uma cópia intermediária tratada (A9) também é uma boa prática de reprodutibilidade que vai facilitar muito quando você for de fato escrever o notebook de análise.
</div>

### 5.2 Análise dos Dados

**A10.** **Estatística descritiva** das variáveis contínuas (`calls_count`, `call_duration`, `total_call_duration`, `wait_time`) — `.describe()`, histogramas e boxplots.

**A11.** **Distribuição de chamadas** por `direction`, `internal` e `is_missed_call` — tabelas de frequência e gráficos de barras.

**A12.** **Ranking de operadores** por taxa de chamadas perdidas e por tempo médio de espera — top 10 piores, com identificação dos operadores associados.

**A13.** **Relação entre tempo de espera e chamadas perdidas** — gráfico de dispersão por operador + coeficiente de correlação.

**A14.** **Análise por plano tarifário** — comparação dos KPIs entre planos A, B e C.

**A15.** **Análise por antiguidade** — correlação entre tempo de casa do cliente e KPIs de seus operadores.

**A16.** **Análise temporal** — evolução diária e semanal do volume de chamadas, taxa de perdidas e tempo médio de espera.

**A17.** **Visualizações finais** consolidadas — heatmap de correlação, boxplots comparativos, séries temporais.

### 5.3 Validação das Hipóteses

Para cada hipótese será escolhido o teste estatístico adequado, com justificativa da escolha com base na distribuição dos dados (normalidade verificada com Shapiro-Wilk ou Kolmogorov-Smirnov) e no tipo de variável.

| Hipótese | Teste | Justificativa |
|---|---|---|
| H1 | Correlação de **Spearman** | Relação monotônica esperada; dados não necessariamente normais |
| H2 | **Mann-Whitney U** ou **t de Student** | Comparação de dois grupos (eficientes vs. ineficientes) |
| H3 | **t de Student** para amostras independentes (ou **Mann-Whitney** se não houver normalidade) | Comparação de médias entre internas e externas |
| H4 | **ANOVA** de um fator (ou **Kruskal-Wallis** se premissas violadas) | Comparação entre 3 planos |
| H5 | Correlação de **Pearson/Spearman** | Associação entre variáveis contínuas |

Para cada teste serão reportados: **estatística do teste**, **valor-p**, **tamanho de efeito** (quando aplicável) e **interpretação no contexto do negócio**. O nível de significância adotado será **α = 0,05**.

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Boa, escolher o teste com base na normalidade dos dados (Shapiro-Wilk/KS) em vez de aplicar um teste padrão pra tudo é exatamente a postura certa aqui, e a tabela com hipótese, teste e justificativa deixa isso bem verificável.
</div>

<div class="alert alert-block alert-warning">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Um ponto que não é ensinado no curso, então fica como opcional: como você vai rodar 5 testes de hipótese diferentes com o mesmo α = 0,05, a chance de achar pelo menos um resultado "significativo" só por acaso sobe (erro do tipo I acumulado). Se quiser deixar a análise mais rigorosa, dá pra ajustar o α com uma correção de Bonferroni ou usar o `statsmodels.stats.multitest.multipletests`, que já faz isso pra uma lista de p-valores:

```python
from statsmodels.stats.multitest import multipletests
rejeitados, p_ajustados, _, _ = multipletests(lista_de_pvalores, alpha=0.05, method='bonferroni')
```

Referência: https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html
</div>

### 5.4 Comunicação dos Resultados

**C1 — Dashboard interativo** (Tableau, Power BI ou Looker Studio) contendo:
- KPIs principais em destaque (taxa de chamadas perdidas, tempo médio de espera, volume de saída)
- Ranking dos operadores ineficientes
- Filtros por plano tarifário, direção, período e cliente
- Séries temporais
- Mapa ou tabela consolidada por operador

**C2 — Relatório final no formato CAR (Challenge, Actions, Results):**
- **Challenge:** problema de negócio, contexto da CallMeMaybe e hipóteses testadas
- **Actions:** metodologia aplicada, tratamento dos dados e abordagem analítica
- **Results:** achados principais, impacto no negócio e recomendações concretas aos supervisores

**C3 — Consolidação:** links do dashboard e do relatório serão incluídos ao final do notebook principal para facilitar o acesso do revisor.

### 5.5 Estimativa de Tempo

Prazo total disponível: **duas semanas**. Estimativa por etapa:

| Etapa | Dias úteis |
|---|---|
| Preparação e tratamento dos dados | 2 |
| Análise Exploratória de Dados (EDA) | 3 |
| Validação das hipóteses (testes estatísticos) | 2 |
| Construção do dashboard | 2 |
| Redação do relatório final (CAR) | 1 |
| Revisões e ajustes finais | 2 |
| **Total** | **12** |

Margem de segurança: **2 dias** para imprevistos ou ajustes solicitados pelo revisor.

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Boa, o formato CAR e a estimativa de tempo com margem de segurança mostram que você pensou no projeto como um trabalho a ser entregue num prazo real, não só como exercício acadêmico. É exatamente esse tipo de organização que se espera de um analista no mercado.
</div>

## 6. Stakeholders Impactados

| Stakeholder | Papel | Como será impactado |
|---|---|---|
| **Supervisores de operadores** | Usuários diretos da nova funcionalidade | Usarão os indicadores para identificar operadores ineficientes e tomar decisões de gestão |
| **Gestores de qualidade** | Responsáveis por treinamento e processos | Poderão direcionar treinamentos com base nos KPIs identificados |
| **Diretoria de Operações** | Decisões estratégicas | Terão visão agregada da eficiência operacional para priorização de investimentos |
| **Clientes (organizações contratantes)** | Beneficiários indiretos | Receberão atendimento mais rápido e eficiente, melhorando a satisfação |
| **Equipe de Produto da CallMeMaybe** | Desenvolvedores da funcionalidade | Usarão os achados para validar e refinar a ferramenta de identificação de operadores ineficientes |

<div class="alert alert-block alert-success">
<b>Comentário do revisor v1 (Gabriel Aguiar):</b> <a class="tocSkip"></a>

Boa, mapear os stakeholders e o que cada um ganha com o resultado da análise deixa claro pra quem esse trabalho serve, não só o que ele calcula. Isso ajuda bastante na hora de escrever o relatório final no formato CAR, porque você já sabe qual recomendação interessa a cada público.
</div>